In [ ]:

import requests
ITEM_ID = 39357569
STORY_ID = 151437

with open('fnf_toks.txt', 'r') as f:
    toks = f.read().splitlines()

api_url = "https://api.fresenia-dev.man.poznan.pl/v2/"
TOKEN = toks[0]

def unpack_item(item):
    if item['success']:
        return item['data']
    else:        
        raise Exception(f"Unsuccessful API call: {item['message']}")

def get_single_resource(resource_type, resource_id):
    headers = {"Authorization": f"Bearer {TOKEN}"}
    endpoint_invoke = f"{api_url}{resource_type}/{resource_id}" if resource_id else f"{api_url}{resource_type}"
    r = requests.get(endpoint_invoke, headers=headers)
    return unpack_item(r.json())

def get_subresource_list(resource_type, resource_id, subresource_type):
    headers = {"Authorization": f"Bearer {TOKEN}"}
    endpoint_invoke = f"{api_url}{resource_type}/{resource_id}/{subresource_type}"
    r = requests.get(endpoint_invoke, headers=headers)
    return unpack_item(r.json())

def this_api_get(endpoint):
    headers = {"Authorization": f"Bearer {TOKEN}"}
    r = requests.get(f"{api_url}{endpoint}", headers=headers)
    return unpack_item(r.json())

def get_item(item_id):
    return get_single_resource("items", item_id)

def get_story(story_id):
    return get_single_resource("stories", story_id)

def get_all_datasets():
    return get_single_resource("datasets", "")

def get_dataset(dataset_id):
    return get_single_resource("datasets", dataset_id)

def get_project(project_id):
    return get_single_resource("projects", project_id)

def get_all_places_from_dataset(dataset_id):
    return get_subresource_list("datasets", dataset_id, "places")

def get_all_places_from_project(project_id):
    return get_subresource_list("projects", project_id, "places")

DUBLIN_DS_ID = 15
DUBLIN_PROJ_ID = 11
# get_item(ITEM_ID)
# get_story(STORY_ID)
# get_all_datasets()
# get_project(DUBLIN_PROJ_ID)
# get_all_places_from_dataset(DUBLIN_DS_ID)

url_for_all_items = 'https://europeana.fresenia-dev.man.poznan.pl/dev/wp-content/themes/transcribathon/requests/api.php/projects/11/places/?limit=3000&orderBy=PlaceId&orderDir=desc&page={PAGE_NUMBER}'
all_res = []
for i in range(1, 12):
    print(f"Getting page {i}...")
    res = unpack_item(requests.get(url_for_all_items.replace("{PAGE_NUMBER}", str(i))).json())
    all_res.extend(res)

DUBLIN_BBOX = [[-7.012024,53.138533],[-5.841980,53.608803]]

min_lon, min_lat = DUBLIN_BBOX[0]
max_lon, max_lat = DUBLIN_BBOX[1]

new_res = [
    item for item in all_res
    if min_lon <= float(item['Longitude']) <= max_lon
    and min_lat <= float(item['Latitude']) <= max_lat
]

print(f"Filtered to {len(new_res)} items within Dublin bbox")

import pandas as pd
dublin_df = pd.DataFrame(new_res)
dublin_map_df = dublin_df[dublin_df['ItemTitle'].str.contains("map", case=False, na=False)]
dublin_map_itms = [get_item(i) for i in dublin_map_df['ItemId'].value_counts().index.tolist()]
dublin_map_itms_df = pd.DataFrame(dublin_map_itms)
dublin_map_itms_df.to_csv("dublin_map_items.csv", index=False)

In [ ]:
import pandas as pd
dublin_map_itms_df = pd.read_csv("dublin_map_items.csv", index_col=False)
dublin_map_itms_df

In [ ]:
dublin_map_itms_df.columns